[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdgordillob/aca_aci_collab/blob/main/notebooks/aca_opt_benchmark.ipynb)

# aca_opt pipeline benchmark

Clones the **source ACI-CO pipeline repo** (`aca_indice_climatico_opt`, not this lightweight companion repo) and runs three of its real pipeline stages -- Stage 1 (merge/resample raw grib to daily), Stage 2 (baseline percentiles), Stage 3 (temperature anomalies) -- directly, as imported Python functions, against one year of raw ERA5 fetched from Drive. Each stage is timed so you can see how this pipeline performs on Colab's hardware and extrapolate to a full 1961--2024 run.

**This is a timing/mechanics smoke test, not a scientifically valid percentile calculation** -- it deliberately uses a single year of data so it finishes quickly. Stage 2 normally runs once over the full 30-year 1961--1990 baseline; see the note in that section for how to extrapolate.

The pipeline scripts needed no changes for this -- `unir_archivos.py`, `calcular_percentil_temperatura.py`, and `calcular_anomalias_temperatura.py` already expose plain, importable functions behind their `__main__` CLI wrappers.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_COLAB

## 1. Get the code

This clones the **pipeline** repo (`aca_indice_climatico_opt`), not `aca_aci_collab` -- the two are separate: this notebook lives in the lightweight companion repo, but it benchmarks the real pipeline.

In [ ]:
import sys, os

REPO_ROOT = "/content/aca_indice_climatico_opt"

if IN_COLAB:
    !git clone --depth 1 https://github.com/mdgordillob/aca_indice_climatico_opt.git {REPO_ROOT}
else:
    REPO_ROOT = os.path.abspath("../../aca_indice_climatico_opt-main")  # adjust if running locally

sys.path.insert(0, os.path.join(REPO_ROOT, "src", "scripts"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src", "utils"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))  # for drive_sync

### Temporary: patch in `drive_sync.py`

`drive_sync.py` isn't pushed to the `aca_indice_climatico_opt` GitHub repo yet, so the clone above doesn't include it. This cell writes it into the freshly-cloned checkout directly so the rest of this notebook can be tested end-to-end right now. **Delete this cell** once `drive_sync.py` is confirmed working and actually pushed -- importing the real, committed copy is what everyone else's clone will get.

In [ ]:
%%writefile {REPO_ROOT}/src/drive_sync.py
"""Sync data from a mounted Google Drive folder into this repo's data/ layout.

Colab-only. The shared Drive folder that backs this repo's data/raw/ and
data/processed/ (confirmed path: "My Drive/2. Datos" -- it's a shortcut
sitting directly in the user's My Drive root; the "Indice Climatico
Actuarial" name seen in Drive's breadcrumb/location panel is the owner's
canonical path, not where the shortcut lives) is shared with specific
named collaborators, not "anyone with the link" -- confirmed from the
Drive UI's "Who has access" panel, which also explains why anonymous
access (tested with gdown) returned HTTP 401. The reliable path is to
mount Drive as a logged-in user who actually has access
(google.colab.drive) and copy/symlink from there.

    from google.colab import drive
    drive.mount("/content/drive")

    import drive_sync
    drive_sync.sync(
        drive_root="/content/drive/MyDrive/2. Datos",
        repo_root="/content/aca_indice_climatico_opt",
    )

Confirmed full structure of "2. Datos" (2026-08-10, via
notebooks/explore_drive_structure.ipynb in the aca_aci_collab repo):

    shapefiles/            48 files, flat -- matches data/shapefiles/
    era5/
      completos/           195 files, flat -- era5_{rain,tmp,wind}_<year>.grib,
                            i.e. the actual raw archive data/raw/era5/ expects
      union/                3 files -- era5_{rain,tmp,wind}_union.nc (final
                            merged daily files, but named differently from
                            what calcular_percentil_*.py hardcodes -- would
                            need renaming, not just copying, to be used directly)
      Combined/             dated snapshots (202412/, 202506/, ...) of the
                            same 3 merged files, named era5_daily_combined_*.nc
                            (matches local naming, but which date is "current"
                            is a judgment call, not automatable)
      percentiles_nc/        dated snapshots (202412/, 202506/, ...) of
                            era5_*_percentil.nc (Stage 2 baseline output --
                            names match data/processed/ exactly)
      otros/, ejemplo_4_73/, "Subsets para excel /"  -- one-off/example
                            files, not part of the steady pipeline layout
    salidas_colombia/       1 file: salidas_colombia.zip (not a flat folder!)
    salidas_cundinamarca_bogota/  1 file: salidas_cundinamarca_bogota.zip
    datos_indice/           dated personal archive (2024-12/, 2025-06/,
                            2025-12/) of anomalias_<region> combined CSVs --
                            does not map 1:1 onto any single local directory,
                            deliberately left out of DEFAULT_MAPPING

Only shapefiles/, era5/completos/, and the two salidas_*.zip are mapped by
default -- the rest (union/, Combined/, percentiles_nc/, datos_indice/) are
dated snapshots or use different filenames than the pipeline scripts
expect, so pulling them in requires a judgment call (which date, rename to
what) that sync() deliberately does not make for you. Pass your own
`mapping=` for those.
"""
import os
import shutil
import zipfile

DEFAULT_MOUNT = "/content/drive"

# Drive subfolder (relative to drive_root) -> local path (relative to repo_root).
# See the module docstring for the full confirmed structure and why the
# other subfolders (union/, Combined/, percentiles_nc/, datos_indice/)
# aren't included here.
DEFAULT_MAPPING = {
    "shapefiles": "data/shapefiles",
    "era5/completos": "data/raw/era5",
    "salidas_colombia": "data/processed/anomalias_colombia",
    "salidas_cundinamarca_bogota": "data/processed/anomalias_cundinamarca_bogota",
}


def ensure_mounted(mount_point=DEFAULT_MOUNT):
    """Mount Google Drive if it isn't already. No-op outside Colab-with-drive-unmounted."""
    if not os.path.isdir(mount_point):
        from google.colab import drive
        drive.mount(mount_point)
    return mount_point


def sync(drive_root, repo_root, mapping=None, mount_point=DEFAULT_MOUNT, copy=True, only=None, extract_zips=True):
    """Copy (or symlink) each configured Drive subfolder into its local data/ path.

    drive_root: path to the shared folder inside the mounted Drive, e.g.
        "/content/drive/MyDrive/2. Datos" -- wherever that folder (or a
        shortcut to it) actually sits in your Drive.
    repo_root: local checkout of the aca_indice_climatico_opt pipeline repo.
    mapping: overrides DEFAULT_MAPPING.
    copy: True copies files (safe -- local runs can't mutate the shared Drive
        folder); False symlinks instead (fast, no duplication, but writes from
        the pipeline would land back in Drive -- only use for read-only work,
        and only for non-zip sources, since symlinking a .zip doesn't extract it).
    only: iterable of mapping keys to sync, e.g. ["era5/completos"], instead
        of everything.
    extract_zips: if the Drive source folder contains one or more .zip files
        (e.g. salidas_colombia/salidas_colombia.zip), extract them into the
        local destination instead of copying the .zip itself. True by default.

    Returns {drive_subfolder: local_path} for whatever was actually synced.
    """
    ensure_mounted(mount_point)
    mapping = mapping or DEFAULT_MAPPING
    if only is not None:
        mapping = {k: v for k, v in mapping.items() if k in only}

    synced = {}
    for drive_sub, local_rel in mapping.items():
        src = os.path.join(drive_root, drive_sub)
        dst = os.path.join(repo_root, local_rel)
        if not os.path.isdir(src):
            print(f"skip {drive_sub!r}: not found under {drive_root}")
            continue

        zip_files = [f for f in os.listdir(src) if f.lower().endswith(".zip")] if extract_zips else []
        if zip_files:
            os.makedirs(dst, exist_ok=True)
            for zf in zip_files:
                with zipfile.ZipFile(os.path.join(src, zf)) as archive:
                    archive.extractall(dst)
                print(f"extracted {drive_sub}/{zf} -> {dst}")
        else:
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if copy:
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                if os.path.islink(dst) or os.path.exists(dst):
                    (os.remove if os.path.islink(dst) else shutil.rmtree)(dst)
                os.symlink(src, dst, target_is_directory=True)
            print(f"synced {drive_sub!r} -> {dst}")

        synced[drive_sub] = dst

    return synced

## 2. Install dependencies

Colab's base image doesn't have `cfgrib`'s system-level `eccodes` library, or `rioxarray`/`geopandas` (used for the shapefile clip step). This repo's own environment already has these, so this cell is a no-op locally.

**Versions are pinned to match this repo's `requirements.txt` exactly** -- an earlier unpinned run reproduced `t_90`/`t_10` bit-for-bit against a local run, but silently dropped the `count_hot`/`count_cold` columns, traced to Colab resolving `xarray==2026.7.0` against this repo's pinned `xarray==2024.11.0` (`calcular_anomalias_temperatura.py`'s `drop_unnecessary_coords()` uses the deprecated `DataArray.drop()`, whose variable/coordinate dispatch has shifted across that version gap). Pinning avoids the mismatch; it doesn't fix the underlying deprecated-API fragility in the script itself, which is a separate, more invasive change.

In [ ]:
if IN_COLAB:
    !apt-get -qq install -y libeccodes-dev > /dev/null
    !pip install -q cfgrib==0.9.14.1 eccodes==2.39.1 rioxarray==0.18.2 geopandas==1.0.1 netCDF4==1.7.2 xarray==2024.11.0

## 3. Mount Drive and fetch one year of raw ERA5

`DRIVE_ROOT` is confirmed to be the right path ("My Drive/2. Datos" -- a shortcut sitting directly in My Drive root; "Indice Climatico Actuarial", seen in Drive's breadcrumb, is the owner's canonical path, not where the shortcut lives) -- edit it only if your own Drive differs. This folder is shared with specific collaborators, not "anyone with the link" (confirmed from the Drive UI's access panel), which is why anonymous fetching isn't available -- this uses `drive.mount()` instead, which authenticates as whoever is logged into this Colab session.

`era5/completos/` is confirmed to hold the flat raw archive -- 195 files named `era5_{rain,tmp,wind}_<year>.grib`, exactly what `data/raw/era5/` needs. Only **one year** of temperature grib is copied locally here (not the whole archive) -- enough for a timing smoke test without a slow full-archive transfer. Change `BENCHMARK_YEAR` to any year you confirm is present.

In [ ]:
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/2. Datos"
BENCHMARK_YEAR = 1985

raw_dir = os.path.join(REPO_ROOT, "data", "raw", "era5")
os.makedirs(raw_dir, exist_ok=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    grib_name = f"era5_tmp_{BENCHMARK_YEAR}.grib"
    src = os.path.join(DRIVE_ROOT, "era5", "completos", grib_name)
    if not os.path.exists(src):
        raise FileNotFoundError(
            f"{src} not found -- pick a BENCHMARK_YEAR present in era5/completos/ "
            "(run explore_drive_structure.ipynb on that subfolder to list them)."
        )

    dst = os.path.join(raw_dir, grib_name)
    shutil.copy(src, dst)
    print(f"copied {dst} ({os.path.getsize(dst) / 1e6:.0f} MB)")

    import drive_sync
    drive_sync.sync(DRIVE_ROOT, REPO_ROOT, only=["shapefiles"])
else:
    print("Not in Colab -- assuming data/raw/era5 and data/shapefiles are already populated locally.")

## 4. Stage 1 -- merge/resample raw grib to daily

`unir_archivos.process_yearly_data_tmp` decodes the hourly GRIB file with `cfgrib` and resamples to daily max/min. This is the expensive, per-year, embarrassingly-parallel step.

In [ ]:
import time
import unir_archivos

stage1_out = os.path.join(REPO_ROOT, "data", "processed", "_benchmark_daily_tmp")
grib_path = os.path.join(raw_dir, f"era5_tmp_{BENCHMARK_YEAR}.grib")

t0 = time.perf_counter()
unir_archivos.process_yearly_data_tmp(grib_path, BENCHMARK_YEAR, "t2m", stage1_out)
unir_archivos.merge_yearly_files(
    stage1_out,
    os.path.join(stage1_out, "era5_daily_combined_tmp_benchmark.nc"),
    "t2m",
)
stage1_time = time.perf_counter() - t0
print(f"Stage 1 (merge/resample, {BENCHMARK_YEAR} only): {stage1_time:.1f}s")

## 5. Stage 2 -- baseline percentiles

`calcular_percentil_temperatura.calcular_percentiles` normally runs once over the full 1961--1990 baseline (30 years) already merged by Stage 1. Here it runs against the single merged year from above instead, purely to time the computation -- the *shape* of the work (groupby-month quantiles over however many years are in the input) is the same; only the year count differs. **Multiply this time by ~30 for a rough estimate of the real baseline computation.**

In [ ]:
import calcular_percentil_temperatura as percentil_tmp

merged_file = os.path.join(stage1_out, "era5_daily_combined_tmp_benchmark.nc")

t0 = time.perf_counter()
estadisticas = percentil_tmp.calcular_percentiles(merged_file)
percentil_tmp.guardar_percentiles(
    estadisticas,
    os.path.join(stage1_out, "percentiles_benchmark.nc"),
    stage1_out,
    guardar_csv=False,
)
stage2_time = time.perf_counter() - t0
print(f"Stage 2 (percentiles, {BENCHMARK_YEAR} only -- not a real 30-year baseline): {stage2_time:.1f}s")

## 6. Stage 3 -- temperature anomalies

`calcular_anomalias_temperatura.procesar_anomalias_temperatura` reads raw grib directly (not Stage 1's output) plus the Stage 2 percentile file, and writes one NetCDF per year-month plus a combined CSV -- the same shape of output `aci_lib` reads as `anomalias_colombia/anomalies_temperature` in the other notebook.

In [ ]:
import calcular_anomalias_temperatura as anomalias_tmp

stage3_out = os.path.join(REPO_ROOT, "data", "processed", "_benchmark_anomalias")
os.makedirs(stage3_out, exist_ok=True)
shapefile_path = os.path.join(REPO_ROOT, "data", "shapefiles", "colombia_4326.shp")

t0 = time.perf_counter()
anomalias_tmp.procesar_anomalias_temperatura(
    archivo_percentiles=os.path.join(stage1_out, "percentiles_benchmark.nc"),
    archivo_comparar_location=raw_dir,
    output_csv_path=os.path.join(stage3_out, "anomalies_temperature_benchmark.csv"),
    shapefile_path=shapefile_path if os.path.exists(shapefile_path) else None,
    output_netcdf=stage3_out,
    use_multiprocessing=False,  # single year -- not worth spinning up a pool
)
stage3_time = time.perf_counter() - t0
print(f"Stage 3 (anomalies, {BENCHMARK_YEAR} only): {stage3_time:.1f}s")

## 7. Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"stage": "1. merge/resample", "years_in_this_run": 1, "seconds": stage1_time},
    {"stage": "2. baseline percentiles", "years_in_this_run": 1, "seconds": stage2_time},
    {"stage": "3. anomalies", "years_in_this_run": 1, "seconds": stage3_time},
])
summary["est_seconds_full_1961_2024"] = summary["seconds"] * 64
summary

Stages 1 and 3 scale roughly linearly with the number of years processed (each year is decoded/resampled independently), so the `est_seconds_full_1961_2024` column is a straightforward extrapolation. Stage 2 in the real pipeline runs once, over a fixed 30-year window, not once per year -- its `est_seconds_full_1961_2024` figure isn't meaningful; see the note in Section 5 instead.